# Mobile Money Agent Network Data Clean-Up

This project cleans and validates mobile money agent transaction data. The pipeline removes duplicates, standardises transaction types, fixes invalid amounts and dates, flags outliers, merges agent information, and produces summary tables for analysis.

In [1]:
%%writefile generate_data.py

import numpy as np
import pandas as pd

rng = np.random.default_rng(44)

agents = pd.DataFrame({
    "agent_id": [
        "A001", "A002", "A003", "A004", "A005",
        "A006", "A007", "A008", "A009", "A010"
    ],
    "agent_name": [
        "Kamau Agency",
        "Achieng Services",
        "Wanjiku Mobile",
        "Otieno Cash Point",
        "Mwangi Agency",
        "Njeri Services",
        "Omondi Mobile",
        "Chebet Cash Point",
        "Mutiso Agency",
        "Ali Services"
    ],
    "county": [
        "Nairobi",
        "Kisumu",
        "Nakuru",
        "Mombasa",
        "Nairobi",
        "Nakuru",
        "Kisumu",
        "Nairobi",
        "Machakos",
        "Mombasa"
    ]
})

n = 300

dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(
    rng.integers(0, 180, n),
    unit="D"
)

txns = pd.DataFrame({
    "txn_id": [f"T{i:04d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(
        [
            "A001", "A002", "A003", "A004", "A005",
            "A006", "A007", "A008", "A009", "A010",
            "A999"
        ],
        n
    ),
    "txn_type": rng.choice(
        ["Deposit", "Withdrawal"],
        n
    ),
    "amount": rng.integers(
        100,
        30000,
        n
    ).astype(object),
    "txn_date": dates.strftime("%Y-%m-%d")
})

txns.loc[5, "txn_type"] = "deposit"
txns.loc[15, "txn_type"] = "WITHDRAWAL"
txns.loc[25, "txn_type"] = " Deposit "

txns.loc[10, "amount"] = "abc"
txns.loc[20, "amount"] = None
txns.loc[30, "amount"] = "12,500"

txns.loc[40, "amount"] = 250000

txns.loc[12, "txn_date"] = "2026/02/15"
txns.loc[22, "txn_date"] = "15-03-2026"

txns = pd.concat(
    [txns, txns.iloc[[50, 100]]],
    ignore_index=True
)

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Generated agent_transactions.csv")
print("Generated agents.csv")
print("Transaction rows:", len(txns))
print("Agent rows:", len(agents))

Overwriting generate_data.py


In [2]:
!python generate_data.py

Generated agent_transactions.csv
Generated agents.csv
Transaction rows: 302
Agent rows: 10


In [3]:
%%writefile cleaning_pipeline.py

import pandas as pd
import numpy as np


def clean_agent_data(txns, agents):
    """
    Clean mobile money transaction data and merge agent details.

    Parameters
    ----------
    txns : pandas.DataFrame
        Raw transaction data.
    agents : pandas.DataFrame
        Agent reference data.

    Returns
    -------
    clean_df : pandas.DataFrame
        Cleaned transaction data with agent county information.
    log : list
        Cleaning log describing each change made.
    """

    log = []
    df = txns.copy()

    # 1. Remove duplicate transaction IDs
    before = len(df)
    df = df.drop_duplicates(subset=["txn_id"]).copy()
    removed_duplicates = before - len(df)
    log.append(f"Removed {removed_duplicates} duplicate transaction rows")

    # 2. Standardise transaction type
    df["txn_type"] = (
        df["txn_type"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    valid_types = ["Deposit", "Withdrawal"]
    invalid_types = (~df["txn_type"].isin(valid_types)).sum()

    df.loc[
        ~df["txn_type"].isin(valid_types),
        "txn_type"
    ] = np.nan

    log.append(
        f"Found {invalid_types} invalid transaction type values"
    )

    # 3. Clean amount column
    df["amount"] = (
        df["amount"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    df["amount"] = pd.to_numeric(
        df["amount"],
        errors="coerce"
    )

    # Flag missing or invalid amounts before filling
    df["amount_imputed"] = df["amount"].isna()

    missing_amounts = int(df["amount_imputed"].sum())

    median_amount = df["amount"].median()

    df["amount"] = df["amount"].fillna(
        median_amount
    )

    log.append(
        f"Imputed {missing_amounts} missing or invalid amounts with the median"
    )

    # 4. Clean transaction dates
    original_dates = df["txn_date"].copy()

    df["txn_date"] = pd.to_datetime(
        df["txn_date"],
        errors="coerce",
        dayfirst=False
    )

    # Retry unresolved dates using day-first parsing
    unresolved = df["txn_date"].isna()

    if unresolved.any():
        df.loc[unresolved, "txn_date"] = pd.to_datetime(
            original_dates[unresolved],
            errors="coerce",
            dayfirst=True
        )

    invalid_dates = int(df["txn_date"].isna().sum())

    log.append(
        f"Found {invalid_dates} transaction dates that could not be parsed"
    )

    # 5. Flag amount outliers using IQR
    q1 = df["amount"].quantile(0.25)
    q3 = df["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    df["is_outlier"] = (
        (df["amount"] < lower_fence)
        | (df["amount"] > upper_fence)
    )

    outlier_count = int(df["is_outlier"].sum())

    log.append(
        f"Flagged {outlier_count} amount outliers using the IQR method"
    )

    # 6. Merge agent information
    before_merge = len(df)

    clean_df = df.merge(
        agents,
        on="agent_id",
        how="left",
        validate="many_to_one"
    )

    after_merge = len(clean_df)

    unknown_agents = int(
        clean_df["county"].isna().sum()
    )

    log.append(
        f"Merged agent data: {before_merge} rows before merge and "
        f"{after_merge} rows after merge"
    )

    log.append(
        f"Found {unknown_agents} transactions with unknown agents"
    )

    # 7. Format dates as ISO YYYY-MM-DD
    clean_df["txn_date"] = clean_df["txn_date"].dt.strftime(
        "%Y-%m-%d"
    )

    return clean_df, log

Overwriting cleaning_pipeline.py


In [4]:
import pandas as pd

from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

clean_df, log = clean_agent_data(txns, agents)

print("Raw transaction rows:", len(txns))
print("Clean transaction rows:", len(clean_df))

print("\nCleaning Log:")
for item in log:
    print("-", item)

Raw transaction rows: 302
Clean transaction rows: 300

Cleaning Log:
- Removed 2 duplicate transaction rows
- Found 0 invalid transaction type values
- Imputed 2 missing or invalid amounts with the median
- Found 1 transaction dates that could not be parsed
- Flagged 1 amount outliers using the IQR method
- Merged agent data: 300 rows before merge and 300 rows after merge
- Found 18 transactions with unknown agents


/content/cleaning_pipeline.py:94: UserWarning: Parsing dates in %Y/%m/%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df.loc[unresolved, "txn_date"] = pd.to_datetime(


In [5]:
audit_table = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": txns.duplicated(subset=["txn_id"]).sum()
    },
    {
        "issue": "Inconsistent transaction type formatting",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows affected": (
            txns["txn_type"].astype(str).str.strip().str.title()
            != txns["txn_type"].astype(str)
        ).sum()
    },
    {
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness / Validity",
        "column": "amount",
        "rows affected": pd.to_numeric(
            txns["amount"].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        ).isna().sum()
    },
    {
        "issue": "Inconsistent or invalid date formats",
        "dimension": "Consistency / Validity",
        "column": "txn_date",
        "rows affected": (
            ~txns["txn_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$")
        ).sum()
    },
    {
        "issue": "Unknown agent IDs",
        "dimension": "Accuracy",
        "column": "agent_id",
        "rows affected": (~txns["agent_id"].isin(agents["agent_id"])).sum()
    },
    {
        "issue": "Amount outliers",
        "dimension": "Validity",
        "column": "amount",
        "rows affected": clean_df["is_outlier"].sum()
    }
])

audit_table

,issue,dimension,column,rows affected
0,Duplicate transaction IDs,Uniqueness,txn_id,2
1,Inconsistent transaction type formatting,Consistency,txn_type,3
2,Missing or invalid amounts,Completeness / Validity,amount,2
3,Inconsistent or invalid date formats,Consistency / Validity,txn_date,2
4,Unknown agent IDs,Accuracy,agent_id,18
5,Amount outliers,Validity,amount,1


In [6]:
rows_before_merge = len(
    txns.drop_duplicates(subset=["txn_id"])
)

rows_after_merge = len(clean_df)

unknown_agents = clean_df["county"].isna().sum()

print("Rows before merge:", rows_before_merge)
print("Rows after merge:", rows_after_merge)
print("Transactions with unknown agents:", unknown_agents)

Rows before merge: 300
Rows after merge: 300
Transactions with unknown agents: 18


In [7]:
county_summary = (
    clean_df.groupby("county", dropna=False)
    .agg(
        total_transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        outliers=("is_outlier", "sum")
    )
    .reset_index()
    .sort_values("total_amount", ascending=False)
)

county_summary

,county,total_transactions,total_amount,average_amount,outliers
3,Nairobi,78,1250963.0,16037.987179,0
0,Kisumu,56,851753.0,15209.875000,0
1,Machakos,36,770898.0,21413.833333,1
4,Nakuru,54,763945.0,14147.129630,0
2,Mombasa,58,745830.0,12859.137931,0
5,NaN,18,317383.0,17632.388889,0


In [8]:
weekly_data = clean_df.copy()

weekly_data["txn_date"] = pd.to_datetime(
    weekly_data["txn_date"]
)

weekly_data["week"] = weekly_data["txn_date"].dt.to_period("W").astype(str)

weekly_pivot = pd.pivot_table(
    weekly_data,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

weekly_pivot

week,2025-12-29/2026-01-04,2026-01-05/2026-01-11,2026-01-12/2026-01-18,2026-01-19/2026-01-25,2026-01-26/2026-02-01,2026-02-02/2026-02-08,2026-02-09/2026-02-15,2026-02-16/2026-02-22,2026-02-23/2026-03-01,2026-03-02/2026-03-08,...,2026-05-04/2026-05-10,2026-05-11/2026-05-17,2026-05-18/2026-05-24,2026-05-25/2026-05-31,2026-06-01/2026-06-07,2026-06-08/2026-06-14,2026-06-15/2026-06-21,2026-06-22/2026-06-28,2026-06-29/2026-07-05,NaT
county,,,,,,,,,,,,,,,,,,,,,
Kisumu,5614.0,12032.0,6079.0,25798.0,52187.0,50030.0,57690.0,116865.0,58866.0,27936.0,...,9593.0,54189.0,0.0,40063.0,46722.0,882.0,33772.0,72407.0,24337.0,0.0
Machakos,38532.0,0.0,31998.0,39029.0,24056.0,7342.0,33303.0,28676.0,15812.0,21531.0,...,26532.0,47860.0,0.0,18003.0,1462.0,8172.0,24469.0,259423.0,0.0,0.0
Mombasa,0.0,52666.0,22533.0,4283.0,0.0,33485.0,0.0,55912.0,16843.0,16558.0,...,83484.0,25093.0,0.0,44756.0,29894.0,39865.0,26524.0,16108.0,0.0,0.0
Nairobi,0.0,16278.0,90497.0,82977.0,32315.0,38227.0,71263.0,12293.0,40634.0,9177.0,...,19797.0,31403.0,101810.0,35138.0,76455.0,58344.0,21399.0,48767.0,0.0,5580.0
Nakuru,0.0,0.0,7498.0,63623.0,71432.0,99400.0,11123.0,16968.0,37933.0,59854.0,...,9119.0,25644.0,66971.0,28531.0,31477.0,34987.0,11056.0,20216.0,0.0,0.0


In [9]:
import os

os.makedirs("outputs", exist_ok=True)

clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Saved outputs/agent_transactions_clean.csv")
print("Rows:", len(clean_df))
print("Duplicate txn_id values:", clean_df["txn_id"].duplicated().sum())
print("Transaction types:", sorted(clean_df["txn_type"].dropna().unique()))

Saved outputs/agent_transactions_clean.csv
Rows: 300
Duplicate txn_id values: 0
Transaction types: ['Deposit', 'Withdrawal']


In [10]:
%%writefile requirements.txt
pandas
numpy

Overwriting requirements.txt


## Data Quality Report

### Completeness

The transaction data contained missing or invalid values in the `amount` column. These values were identified before cleaning, flagged using the `amount_imputed` column, and then replaced with the median transaction amount. This preserved the affected rows while making the amount column usable for analysis.

### Validity

The `amount` column contained invalid text values and one unusually large transaction. Invalid amounts were converted to missing values using `pd.to_numeric(..., errors="coerce")` and then imputed with the median. Outliers were detected using the IQR method and recorded in the `is_outlier` column rather than deleted automatically.

### Accuracy

Some transactions referenced agent IDs that were not present in the agent reference table. A left merge was used so that these transactions were not lost. Transactions with unmatched agents were identified by missing county values and reported during post-merge validation.

### Consistency

Transaction type values appeared in inconsistent formats such as lowercase, uppercase and values containing extra spaces. These were standardised to `Deposit` and `Withdrawal`. Transaction dates also appeared in more than one format and were converted to ISO format `YYYY-MM-DD`.

### Uniqueness

The raw transaction file contained duplicate transaction IDs. Duplicate records were removed using `txn_id` as the transaction key. The cleaned output was checked to confirm that no duplicate `txn_id` values remained.

### Timeliness

The generated dataset covers transactions from January to June 2026. Dates were converted into a consistent datetime format so that transactions could be grouped correctly by week and used in the weekly pivot table.

### Final Result

The cleaning pipeline produces a reusable cleaned dataset together with a log of all cleaning actions. It preserves valid transactions, flags values that require review, merges county information from the agent table and creates data suitable for summary analysis.

In [11]:
%%writefile README.md
# Mobile Money Agent Network Data Clean-Up

## Project Summary

This project cleans and validates a mobile money agent transaction dataset. The workflow removes duplicate transaction IDs, standardises transaction types, fixes invalid and missing amounts, converts transaction dates into ISO format, flags outliers using the IQR method, merges agent county information, and produces summary tables for analysis.

## How to Regenerate the Data

Run:

python generate_data.py

This creates the raw transaction and agent reference CSV files using random seed 44.

## How to Run the Cleaning Pipeline

Open `agent_cleanup.ipynb` in Google Colab or Jupyter Notebook and run all cells from top to bottom.

The notebook imports and calls `clean_agent_data()` from `cleaning_pipeline.py`.

## Before and After Row Count

The generated raw transaction dataset contains 302 rows.

After removing duplicate transaction IDs, the cleaned dataset contains 300 rows.

## Output

The cleaned dataset is saved to:

`outputs/agent_transactions_clean.csv`

The final output contains no duplicate transaction IDs, standardised transaction types, numeric amounts, ISO-formatted transaction dates, the `amount_imputed` and `is_outlier` flag columns, and agent county information.

## Data Quality Checks

The notebook includes:

- A data-quality audit table
- Post-merge row-count validation
- Unknown-agent checks
- A county summary table
- A weekly pivot table
- A Data Quality Report organised by the six dimensions of data quality

Overwriting README.md
